# 1.0版本

In [3]:
# coding: utf-8
import os, gc, glob, json, logging, csv
import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from collections import defaultdict
from v1_0_20251201.run_model import generate_score

# =====================================================
# 1. 日志控制
# =====================================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

# =====================================================
# 🔧 统一任务参数（只改这里）
# =====================================================
TASK_NAME = "risk000006test"
BASE_DIR = "/opt/workspace/aus_group_drive/Eliam/cdaml_bigq"

# =====================================================
# 2. 路径配置
# =====================================================
SAMPLE_PATH = f"{BASE_DIR}/data/{TASK_NAME}/samples_test.csv"
TXN_DIR = f"{BASE_DIR}/modeling/tmp_eliam_{TASK_NAME}_variable_illion_transaction_middle"
BAL_DIR = f"{BASE_DIR}/modeling/tmp_eliam_{TASK_NAME}_balance_time_series_middle"
OUT_PATH = f"{BASE_DIR}/data/{TASK_NAME}/sample_with_feature.csv"
ERROR_PATH = f"{BASE_DIR}/data/{TASK_NAME}/sample_with_feature_error_detail.csv"

CHUNK_SIZE = 500_000
WRITE_BATCH = 200  # 批量写入条数（按磁盘情况可调大）
POOL_CHUNK = 50    # pool.imap_unordered 的chunksize（可调）

# =====================================================
# 3. 工具函数
# =====================================================
def is_valid_float(x):
    try:
        float(x)
        return True
    except Exception:
        return False

def is_valid_balance_record(rec: dict) -> bool:
    # 保留你原本逻辑：balance 至少有一个可转 float
    return any(is_valid_float(rec.get(f)) for f in ["balance"])

# =====================================================
# 4. 索引构建（优化：drop_duplicates + to_numpy，避免逐行循环）
# =====================================================
def build_csv_index(csv_dir: str):
    logger.info(f"📚 构建索引：{csv_dir}")
    index = defaultdict(set)

    fps = glob.glob(os.path.join(csv_dir, "*.csv"))
    for fp in fps:
        try:
            for chunk in pd.read_csv(fp, usecols=["user_id", "application_id"], chunksize=CHUNK_SIZE):
                uniq = chunk.drop_duplicates(subset=["user_id", "application_id"])
                for uid, aid in uniq[["user_id", "application_id"]].to_numpy():
                    index[(uid, aid)].add(fp)
        except Exception as e:
            logger.warning(f"⚠️ 索引失败跳过：{fp} | {e}")

    logger.info(f"✅ 索引完成：{len(index)} keys")
    return index

# =====================================================
# 5. 读取匹配行（优化：usecols + chunksize，只读必要列）
# =====================================================
TXN_USECOLS = [
    "user_id", "application_id",
    "amount", "balance", "bank_account_id", "category",
    "dr_cr", "illion_trx_uuid", "text", "third_party",
    "transaction_date", "transaction_id", "trx_type"
]

BAL_USECOLS = [
    "user_id", "application_id",
    "job_id", "balance", "balance_date", "balance_id", "bank_account_id"
]

def load_match_rows_from_index(csv_index, user_id, application_id, usecols):
    fps = csv_index.get((user_id, application_id))
    if not fps:
        return pd.DataFrame(columns=usecols)

    matched = []
    for fp in fps:
        try:
            for chunk in pd.read_csv(fp, usecols=usecols, chunksize=CHUNK_SIZE):
                m = chunk[(chunk["user_id"] == user_id) & (chunk["application_id"] == application_id)]
                if not m.empty:
                    matched.append(m)
        except Exception:
            continue

    return pd.concat(matched, ignore_index=True) if matched else pd.DataFrame(columns=usecols)

# =====================================================
# 6. 构造模型输入（优化：fillna("") 替代 applymap）
# =====================================================
def build_input_data(r, txn_index, bal_index):
    uid, aid = r["user_id"], r["application_id"]
    ft = str(r["sample_datetime"])

    txn_df = load_match_rows_from_index(txn_index, uid, aid, TXN_USECOLS)
    bal_df = load_match_rows_from_index(bal_index, uid, aid, BAL_USECOLS)

    txn_records = []
    if not txn_df.empty:
        txn_records = (
            txn_df[
                [
                    "amount", "balance", "bank_account_id", "category",
                    "dr_cr", "illion_trx_uuid", "text", "third_party",
                    "transaction_date", "transaction_id", "trx_type"
                ]
            ]
            .fillna("")
            .to_dict("records")
        )

    bal_records = []
    if not bal_df.empty:
        raw_bal = (
            bal_df[["job_id", "balance", "balance_date", "balance_id", "bank_account_id"]]
            .fillna("")
            .to_dict("records")
        )
        for rec in raw_bal:
            if is_valid_balance_record(rec):
                bal_records.append(rec)

    return {
        "userId": uid,
        "applicationId": aid,
        "flowTime": ft,
        "illion_raw_transactions": txn_records,
        "illion_day_end_balances": bal_records
    }

# =====================================================
# 7. 多进程全局索引（避免每个任务都pickle大对象）
# =====================================================
_TXN_INDEX = None
_BAL_INDEX = None

def _init_worker(txn_index, bal_index):
    global _TXN_INDEX, _BAL_INDEX
    _TXN_INDEX = txn_index
    _BAL_INDEX = bal_index

# =====================================================
# 8. 单样本执行（fail fast，但不中断）
# =====================================================
def process_one_sample(r):
    global _TXN_INDEX, _BAL_INDEX
    try:
        input_data = build_input_data(r, _TXN_INDEX, _BAL_INDEX)
        res = generate_score(input_vars=input_data)

        out = dict(r)
        if isinstance(res, dict):
            for k, v in res.items():
                # ❌ 彻底丢弃 *_features 这个字段名
                if k.endswith("_features"):
                    if isinstance(v, dict):
                        out.update(v)   # 只展开特征
                    # v 是 None 的情况：直接跳过
                    continue

                out[k] = v
        return out, None

    except Exception as e:
        out = dict(r)
        out["feature_error"] = str(e)

        err_detail = {
            "user_id": r.get("user_id"),
            "application_id": r.get("application_id"),
            "error_message": str(e),
            "input_data_json": json.dumps(
                build_input_data(r, _TXN_INDEX, _BAL_INDEX),
                ensure_ascii=False
            )
        }
        return out, err_detail

    finally:
        gc.collect()

# =====================================================
# 9. 批量写入（csv.DictWriter，避免每条 DataFrame.to_csv 的极慢写法）
# =====================================================
def _ensure_writer(f, fieldnames, write_header):
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    if write_header:
        w.writeheader()
    return w

def main():
    logger.info("🚀 加载 samples")
    df = pd.read_csv(SAMPLE_PATH)
    df["sample_datetime"] = df["sample_datetime"].astype(str)

    done_keys = set()
    if os.path.exists(OUT_PATH):
        logger.info("🔁 启用断点续跑")
        done_df = pd.read_csv(OUT_PATH, usecols=["user_id", "application_id"])
        done_keys = set(zip(done_df.user_id, done_df.application_id))

    records = [r for r in df.to_dict("records") if (r["user_id"], r["application_id"]) not in done_keys]
    if not records:
        logger.info("✅ 无待跑样本")
        return

    # 一次性构建索引（大头优化）
    txn_index = build_csv_index(TXN_DIR)
    bal_index = build_csv_index(BAL_DIR)

    n_workers = max(cpu_count() - 4, 1)
    write_header = not os.path.exists(OUT_PATH)
    write_err_header = not os.path.exists(ERROR_PATH)

    base_out_fields = list(df.columns)
    if "feature_error" not in base_out_fields:
        base_out_fields.append("feature_error")
    out_fieldnames = base_out_fields[:]  # 动态扩展
    err_fields = ["user_id", "application_id", "error_message", "input_data_json"]

    out_buffer, err_buffer = [], []

    with open(OUT_PATH, "a", encoding="utf-8", newline="") as fout, \
         open(ERROR_PATH, "a", encoding="utf-8", newline="") as ferr:

        out_writer = None
        err_writer = _ensure_writer(ferr, err_fields, write_err_header)
        write_err_header = False

        with Pool(n_workers, initializer=_init_worker, initargs=(txn_index, bal_index)) as pool:
            for out, err_detail in tqdm(
                pool.imap_unordered(process_one_sample, records, chunksize=POOL_CHUNK),
                total=len(records),
                desc="Processing samples"
            ):
                # 字段动态扩展（保留“所有功能”：新特征列也能写出来）
                for k in out.keys():
                    if k not in out_fieldnames:
                        out_fieldnames.append(k)

                out_buffer.append(out)
                if err_detail:
                    err_buffer.append(err_detail)

                if len(out_buffer) >= WRITE_BATCH:
                    if out_writer is None:
                        out_writer = _ensure_writer(fout, out_fieldnames, write_header)
                        write_header = False
                    if out_writer.fieldnames != out_fieldnames:
                        out_writer = csv.DictWriter(fout, fieldnames=out_fieldnames, extrasaction="ignore")

                    out_writer.writerows(out_buffer)
                    fout.flush()
                    out_buffer.clear()

                    if err_buffer:
                        err_writer.writerows(err_buffer)
                        ferr.flush()
                        err_buffer.clear()

        # flush尾巴
        if out_buffer:
            if out_writer is None:
                out_writer = _ensure_writer(fout, out_fieldnames, write_header)
                write_header = False
            if out_writer.fieldnames != out_fieldnames:
                out_writer = csv.DictWriter(fout, fieldnames=out_fieldnames, extrasaction="ignore")
            out_writer.writerows(out_buffer)
            fout.flush()

        if err_buffer:
            err_writer.writerows(err_buffer)
            ferr.flush()

    logger.info(f"✅ 完成：{OUT_PATH}")
    logger.info(f"⚠️ 错误明细：{ERROR_PATH}")

if __name__ == "__main__":
    main()

2026-01-05 04:44:37,191 | INFO | 🚀 加载 samples
2026-01-05 04:44:37,199 | INFO | 🔁 启用断点续跑
2026-01-05 04:44:37,203 | INFO | 📚 构建索引：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/tmp_eliam_risk000006test_variable_illion_transaction_middle
2026-01-05 04:44:45,267 | INFO | ✅ 索引完成：6224 keys
2026-01-05 04:44:45,268 | INFO | 📚 构建索引：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/tmp_eliam_risk000006test_balance_time_series_middle
2026-01-05 04:44:46,863 | INFO | ✅ 索引完成：6226 keys
Processing samples: 100%|██████████| 30/30 [01:21<00:00,  2.71s/it]
2026-01-05 04:46:08,510 | INFO | ✅ 完成：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/data/risk000006test/sample_with_feature.csv
2026-01-05 04:46:08,512 | INFO | ⚠️ 错误明细：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/data/risk000006test/sample_with_feature_error_detail.csv


# 2.0 

In [1]:
from score_traceback import run_score_pipeline

base_dir = "/opt/workspace/aus_group_drive/Eliam/cdaml_bigq"
task_name = "risk000006test"

run_score_pipeline(
    sample_path=f"{base_dir}/data/{task_name}/samples.csv",
    txn_dir=f"{base_dir}/modeling/tmp_eliam_{task_name}_variable_illion_transaction_middle",
    bal_dir=f"{base_dir}/modeling/tmp_eliam_{task_name}_balance_time_series_middle",
    out_path=f"{base_dir}/data/{task_name}/sample_with_feature.csv",
    err_path=f"{base_dir}/data/{task_name}/sample_with_feature_error_detail.csv",
)

2026-01-05 04:54:12,685 | INFO | 🚀 加载 samples
2026-01-05 04:54:12,848 | INFO | 📚 构建索引：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/tmp_eliam_risk000006test_variable_illion_transaction_middle
2026-01-05 04:54:20,820 | INFO | ✅ 索引完成：6224 keys
2026-01-05 04:54:20,821 | INFO | 📚 构建索引：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/tmp_eliam_risk000006test_balance_time_series_middle
2026-01-05 04:54:22,548 | INFO | ✅ 索引完成：6226 keys
Processing samples: 100%|██████████| 6226/6226 [19:45<00:00,  5.25it/s]    
2026-01-05 05:14:08,503 | INFO | ✅ 完成：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/data/risk000006test/sample_with_feature.csv
2026-01-05 05:14:08,504 | INFO | ⚠️ 错误明细：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/data/risk000006test/sample_with_feature_error_detail.csv
